<a href="https://colab.research.google.com/github/vkjadon/hugging_face/blob/main/hf_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Enable GPU Runtime

Navigate to Runtime → Change runtime type → T4 GPU → Save.



Verify GPU access:

In [ ]:
!nvidia-smi

## Install Libraries

The -q flag suppresses verbose output.

In [ ]:
!pip install -q transformers datasets huggingface_hub accelerate

## Authentication
For private models or pushing to Hub, authenticate with your token:

In [ ]:
from huggingface_hub import login
login()  # Opens interactive prompt

For a cleaner workflow, use Colab secrets:

To set up secrets, click the key icon in Colab's left sidebar and add HF_TOKEN with your Hugging Face access token.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
from huggingface_hub import list_datasets

# Get all available datasets
all_datasets = list_datasets()

# Search for specific datasets by name
imdb_datasets = list_datasets(search="imdb")
print(imdb_datasets)
print(dir(imdb_datasets))

In [ ]:
for dataset in imdb_datasets:
    print(dataset.id)

In [ ]:
import pprint

# Get the first item from the generator
first_item = next(list_datasets(search="imdb"))

# Print all available fields and methods
print(dir(first_item))


# Convert the object's attributes to a readable dictionary
pprint.pprint(vars(first_item))


Download full dataset

In [ ]:
from datasets import load_dataset

In [ ]:
help(load_dataset)

The load_dataset function loads datasets from the Hugging Face Hub or local files, with path serving as the mandatory identifier for the dataset repository or file format. Key arguments include split for requesting specific data portions, data_files for mapping local files, and streaming to enable on-the-fly data loading without downloading the full dataset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

This is a DatasetDict object, which acts like a Python dictionary organizing your dataset into three separate parts (splits):

* train: 25,000 labeled rows used to train your model.
* test: 25,000 labeled rows used to evaluate your model's performance.
* unsupervised: 50,000 unlabeled rows meant for extra pre-training or self-supervised learning.
* features: Every single row across all splits contains exactly two columns: text (the movie review string) and label (the sentiment score).

Would you like to see how to access a specific row of text from the training split, or see how the labels are mapped to positive and negative?



Download a subset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb", split={"train" : "train[:1000]", "test" : "test[:1000]"})
print(dataset)

In [ ]:
dataset = load_dataset("stanfordnlp/imdb", split=["train[:1000]", "test[:1000]"])
print(dataset)

Pass a list of strings to the split argument. This returns a standard Python list containing the individual dataset splits in the same order.

In [ ]:
train_data = dataset["train"]
example = train_data[0]
print(example)

In [ ]:
for example in train_data["label"]:
    print(example)

In [ ]:
texts = train_data["text"]

print(texts[:5])

Load a Model Pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", device=0)  # device=0 uses GPU
result = classifier("I love this course!")
print(result)

Load a Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:100]")
print(dataset[0])

Check Device Placement


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

## Memory Management
Colab's free tier provides approximately 15GB of GPU memory on the T4. Use these techniques to work within that limit.

In [ ]:
import torch
import gc

del model  # Delete the model variable
gc.collect()
torch.cuda.empty_cache()

##Load in Lower Precision

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)

##Mount Google Drive
Save models and checkpoints to persist across sessions:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save model
model.save_pretrained('/content/drive/MyDrive/my_model')

Download Files Locally

In [ ]:
from google.colab import files
files.download('output.csv')

In [ ]:
# 2. Search for models
from huggingface_hub import list_models

models = list(list_models(pipeline_tag="text-classification", sort="downloads", limit=5))
for m in models:
    print(f"{m.id}: {m.downloads:,} downloads")

# 3. Try the top model
from transformers import pipeline

In [ ]:
classifier = pipeline("text-classification", model=models[0].id, device=0)
print(classifier("This is amazing!"))

In [ ]:
classifier = pipeline("text-classification", model=models[2].id, device=0)
print(classifier("This is amazing!"))

## Streaming

In [ ]:
from datasets import load_dataset

In [ ]:
def load_and_stream_filter(dataset_name: str, split_name: str, keyword: str):
    """Stream a dataset split and collect up to 5 'text' values containing the keyword.

    Args:
        dataset_name: The name or path of the dataset on the Hub.
        split_name: The name of the split to stream (e.g., 'train').
        keyword: Substring to search for in the 'text' field.

    Returns:
        A list of up to 5 strings from the 'text' column that contain the keyword.
    """
    # 1. Enable streaming to avoid downloading the entire dataset
    dataset = load_dataset(dataset_name, split=split_name, streaming=True)

    results = []

    # 2. Iterate over the stream
    for item in dataset:
        text_content = item.get("text", "")

        # 3. Check for the keyword substring
        if keyword in text_content:
            results.append(text_content)

            # Break early once we hit the 5-string limit
            if len(results) == 5:
                break

    # 4. Return the filtered list
    return results


## Search for positive sentiment markers in movie reviews

In [ ]:
results = load_and_stream_filter(dataset_name="rotten_tomatoes", split_name="train", keyword="masterpiece")

In [ ]:
print(f"Found {len(results)} matches:")
for i, text in enumerate(results, 1):
    print(f"{i}. {text}")

## Optimized Implementation with .filter() and .take()

Hugging Face IterableDataset objects have a built-in .take(n) method. It automatically limits the stream to the first n elements. However, because you need to filter by a keyword before counting to 5, you must combine it with .filter(). If you use .take(5) first, you will only look at the first 5 rows of the dataset, and if they don't contain the keyword, your function will return nothing.

Here is how you can rewrite the function cleanly using the library's core streaming operations:

In [ ]:
def load_and_stream_filter(dataset_name: str, split_name: str, keyword: str):
    # 1. Load the streaming dataset
    dataset = load_dataset(dataset_name, split=split_name, streaming=True)

    # 2. Apply a streaming filter (lazy evaluation)
    filtered_stream = dataset.filter(lambda item: keyword in item.get("text", ""))

    # 3. Limit the stream to the first 5 matches and extract the 'text' field
    # We use a list comprehension to pull the items out of the stream
    results = [item["text"] for item in filtered_stream.take(5)]

    return results

In [ ]:
results = load_and_stream_filter(dataset_name="rotten_tomatoes", split_name="train", keyword="masterpiece")

In [ ]:
import gradio as gr
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

def predict(text):
    return classifier(text)

demo = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="json"
)

demo.launch()

In [ ]:
import gradio as gr

# Define the function your UI will run
def analyze_review(text):
    word_count = len(text.split())
    sentiment = "Positive Tone 😃" if "good" in text.lower() or "great" in text.lower() else "Neutral/Negative Tone 😐"
    return f"Word Count: {word_count}", f"Predicted Sentiment: {sentiment}"

# Build the UI layout
demo = gr.Interface(
    fn=analyze_review,
    inputs=gr.Textbox(lines=3, placeholder="Enter a movie review here..."),
    outputs=["text", "text"],
    title="IMDB Review Analyzer"
)

# Launch the app inside Colab
demo.launch(share=True)
